# Transfer Learning & Fine-Tuning

#### E/21/087 -  Dewagedara D.M.E.S.

### Task 1 — Use Pretrained CNN as Fixed Feature Extractor

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.utils.data import Subset

# Transforms (ResNet18 needs 224x224)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Dataset (CIFAR-10)
train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

# Model Setup (Task 1: Frozen)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

100%|██████████| 170M/170M [00:05<00:00, 30.5MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 116MB/s]


In [5]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Train Acc: {train_acc:.2f}% | Loss: {loss.item():.4f}")

# Final Validation Check
model.eval()
val_correct = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        val_correct += predicted.eq(labels).sum().item()

print(f"\nFinal Validation Accuracy: {100 * val_correct / len(val_set):.2f}%")

Epoch 1 | Train Acc: 77.66% | Loss: 1.1040
Epoch 2 | Train Acc: 78.86% | Loss: 0.7240

Final Validation Accuracy: 75.90%


#### Answer -

#### 1. Performance Results
Based on the execution output, the ResNet18 model (frozen) achieved the following performance on the dataset:

| Metric                         | Value  |
|--------------------------------|--------|
| Training Accuracy (Epoch 2)    | 78.86% |
| Final Validation Accuracy      | 75.90% |
| Final Training Loss            | 0.7240 |


#### 2. Why Frozen Layers Act as Generic Feature Extractors
When a convolutional neural network (CNN) is trained on a large-scale dataset, it learns a hierarchical representation of visual features:

 - Early layers detect simple and generic patterns such as edges, colors, and textures, which are common across most images.

 - Middle layers combine these low-level features into more meaningful shapes and patterns.

 - Deeper layers learn complex and task-specific features related to particular object categories.

By freezing the pretrained layers, the network retains these learned representations and uses them as a fixed feature extractor. In this approach, the pretrained CNN transforms input images into high-level feature vectors, while only the final classifier layer is trained to map these features to the target dataset classes. This reduces training time and helps prevent overfitting, especially when the new dataset is small.

### Task 2 — Fine-Tune the Last Block

In [6]:
# Unfreeze the last convolutional block (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

# Setup Optimizer with a smaller learning rate for fine-tuning
optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])


In [7]:
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    print(f"Fine-Tuning Epoch {epoch+1} | Acc: {100 * correct / total:.2f}%")

# Final Validation for Task 2
model.eval()
val_correct = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        val_correct += predicted.eq(labels).sum().item()

print(f"\nTask 2 Final Validation Accuracy: {100 * val_correct / len(val_set):.2f}%")

Fine-Tuning Epoch 1 | Acc: 81.40%
Fine-Tuning Epoch 2 | Acc: 86.96%

Task 2 Final Validation Accuracy: 81.30%


#### Answer -

##### 1. New Performance Results

By unfreezing **layer4** and applying a differential learning rate, the model performance improved significantly compared to the fixed feature extractor approach.

| Metric              | Task 1 (Fixed) | Task 2 (Fine-Tuned) | Difference |
|---------------------|----------------|---------------------|------------|
| Training Accuracy   | 78.86%         | 86.96%              | +8.10%     |
| Validation Accuracy | 75.90%         | 81.30%              | +5.40%     |

The results show that fine-tuning higher-level convolution layers enables the model to adapt better to dataset-specific features.


##### 2. Changes in Convergence Rate

- **Faster Accuracy Improvement:**  
  During fine-tuning, the validation accuracy in Epoch 1 (81.40%) already exceeded the final validation accuracy obtained in Task 1 (75.90%). This occurred because the model was already pre-trained and partially optimized during Task 1. Unfreezing **layer4** allowed the network to refine high-level feature representations specific to the dataset.

- **Increased Computational Cost:**  
  Each epoch required slightly more computation compared to Task 1 because gradients were calculated not only for the classifier layer but also for the convolutional filters in **layer4**.

##### 3. Overfitting Indicators

- **Widening Accuracy Gap:**  
  In Task 1, the gap between training and validation accuracy was approximately **2.96%**, whereas in Task 2 this gap increased to approximately **5.66%**.

- **Analysis:**  
  This widening gap suggests the early stages of **overfitting**. When additional layers are unfrozen, the model gains more capacity and may begin to memorize training data patterns that do not fully generalize to the validation dataset.

##### 4. Conclusion

Fine-tuning significantly improved validation accuracy from **75.90%** to **81.30%**, demonstrating the benefit of adapting higher-level feature layers. However, the increased gap between training and validation accuracy indicates potential overfitting if training continues for many epochs. Regularization techniques such as data augmentation or early stopping could be applied to improve generalization.

### Task 3 — Compare & Interpret


##### 1. Comparison Table

| Feature              | Fixed-Feature (Task 1) | Partial Fine-Tuning (Task 2) |
|----------------------|------------------------|------------------------------|
| Final Validation Accuracy | 75.90% | 81.30% |
| Train/Validation Gap | ~3% (Low) | ~6% (Higher) |
| Stability | High (fewer trainable parameters) | Moderate (sensitive to learning rate) |
| Training Speed | Faster | Slower |

##### 2. Training Behavior and Stability

**Accuracy:**  
Partial fine-tuning achieved higher validation accuracy than the fixed-feature approach. By unfreezing **layer4**, the model was able to adapt high-level feature representations to dataset-specific textures and shapes.

**Stability:**  
Task 1 was highly stable because the convolutional layers were frozen, preventing large weight updates and preserving pretrained knowledge. In Task 2, stability depended heavily on the learning rate. If the learning rate is too high, the model may suffer from **catastrophic forgetting**, where pretrained weights are overwritten.

**Convergence:**  
Fine-tuning showed a faster improvement in accuracy during early epochs because the model was already pretrained. However, it also showed a stronger tendency toward overfitting, as indicated by the increased gap between training and validation accuracy.

##### 3. When to Use Each Strategy

##### Fixed-Feature Extraction (Freezing)

Fixed-feature extraction works best in the following scenarios:

- **Small Dataset:**  
  When only a limited number of training images are available, freezing helps prevent overfitting.

- **Domain Similarity:**  
  If the dataset is visually similar to natural images (e.g., common objects or scenes), pretrained features are already suitable.

- **Low Computational Resources:**  
  Training is faster because fewer parameters are updated.


##### Partial Fine-Tuning

Fine-tuning becomes necessary under the following conditions:

- **Domain Shift:**  
  When the dataset differs significantly from natural images (e.g., medical images or specialized textures), deeper layers must adapt.

- **Sufficient Training Data:**  
  More data helps the model learn generalizable patterns when additional layers are unfrozen.

- **Performance Improvement Requirement:**  
  Fine-tuning is useful when validation accuracy plateaus using a fixed-feature extractor and further improvement is needed.

##### 4. Conclusion

The experimental results show that partial fine-tuning improves validation accuracy compared to the fixed-feature approach. However, this improvement comes with increased computational cost and a higher risk of overfitting. Therefore, the choice between freezing and fine-tuning depends on dataset size, domain similarity, and available computational resources.

### Final Reflection

Transfer learning speeds up model development by reusing knowledge from a pretrained network, allowing the model to start with already learned low-level and high-level features (such as edges, textures, and object patterns) instead of training millions of parameters from scratch. This significantly reduces training time, computational cost, and the amount of data required, often enabling good performance within a few epochs even on limited hardware. However, transfer learning may fail when there is a large domain mismatch between the source and target datasets (negative transfer), where pretrained features are not relevant to the new task, such as applying natural-image features to medical or grayscale imagery. It can also fail due to catastrophic forgetting during fine-tuning if an excessively high learning rate overwrites useful pretrained weights, reducing generalization and stability.